<a href="https://colab.research.google.com/github/jazaineam1/BigData2026/blob/main/Cuadernos/6_Neo4j_Contexto_Relacional.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir S6 en Colab"></a>

**Acceso público:** [página del curso](https://jazaineam1.github.io/BigData2026/)

# Sesión 6 — De la fila priorizada al contexto relacional con Neo4j

## Universidad Central
> ### Facultad de Ingeniería y Ciencias Básicas
> ### Maestría en Analítica de Datos — BIG DATA (64491093)

**Caso conductor:** Compras Claras  
**Pregunta profesional:** **Laura ya sabe qué proceso revisar primero. Antes de asignarlo a un auditor, ¿qué relaciones alrededor de ese proceso necesita ver para comprender su contexto?**

### Producto observable

Al terminar tendrás una **ficha relacional de revisión** con:

1. el proceso que llega desde S5;
2. el contexto de prensa heredado;
3. procesos históricos adjudicados de su entidad;
4. proveedores y otras entidades conectadas cuando el dato lo sostenga;
5. una comprobación pandas ↔ Neo4j;
6. una decisión de modelado y una alternativa descartada;
7. un límite concreto;
8. `s06_contexto_procesos.jsonl`, entrada de la siguiente sesión.

## El hilo del evaluador

```text
S3  evidencia documental
 ↓
S4  persistencia compartida en Atlas
 ↓
S5  qué revisar primero → bandeja operacional + ancla elegida
 ↓
S6  qué hay alrededor de lo que Laura va a revisar
```

**PARA LLEVAR.** Neo4j no aparece porque “toca grafos”. Aparece porque la relación entre actores ya forma parte de la pregunta.

## Mapa de la sesión

| Bloque | Pregunta | Qué queda |
|---|---|---|
| 1. Recuperar el ancla | ¿qué proceso llega desde S5? | proceso + entidad + prensa |
| 2. Preparar contexto | ¿qué historial rodea esa entidad? | tabla de contraste |
| 3. Diseñar | ¿qué es nodo y qué es relación? | Entidad → Proceso → Proveedor |
| 4. Contrato pandas | ¿qué debe responder el grafo? | resultado esperado |
| 5. AuraDB | ¿cómo levantamos el servicio? | conexión real |
| 6. Cypher | ¿cómo cargamos y recorremos relaciones? | grafo consultable |
| 7. Verificar | ¿Neo4j conserva la respuesta? | pandas = Neo4j |
| 8. Hito | ¿qué puede sostener Laura? | ficha + límite + export |

In [ ]:
#@title Preparar interactividad { display-mode: "form" }
import base64, json, html as html_lib
from IPython.display import display, HTML

def pregunta_codificada(token):
    p = json.loads(base64.b64decode(token).decode("utf-8"))
    uid = f"s06-p{p['numero']}"
    opts = "".join(
        f'<label style="display:block;margin:8px 0"><input type="radio" name="{uid}" value="{i}"> {html_lib.escape(op)}</label>'
        for i, op in enumerate(p["opciones"])
    )
    retro = json.dumps(p["retro"], ensure_ascii=False)
    box = (
        f'<div style="border:2px solid #175c3c;background:#f4faf6;color:#172019;border-radius:12px;padding:15px;margin:14px 0">'
        f'<strong>Pregunta {p["numero"]} · {html_lib.escape(p["tema"])}</strong>'
        f'<p>{html_lib.escape(p["pregunta"])}</p>{opts}'
        f'<button onclick="(function(){{const e=document.querySelector(\'input[name={uid}]:checked\');'
        f'const s=document.getElementById(\'r-{uid}\');if(!e){{s.textContent=\'Selecciona una opción.\';return;}}'
        f'const i=Number(e.value),r={retro};const ok=i==={p["correcta"]};'
        f's.innerHTML=\'<div style=&quot;margin-top:8px;padding:8px;border-radius:7px;background:#ffffff;color:#172019;border:1px solid #c7d8cd&quot;><strong>\'+(ok?\'Correcto. \':\'Revisa. \')+\'</strong>\'+r[i]+\'</div>\';}})()" '
        f'style="background:#175c3c;color:white;border:0;border-radius:7px;padding:8px 12px">Verificar</button>'
        f'<div id="r-{uid}" aria-live="polite"></div></div>'
    )
    display(HTML(box))

def tutorial(url, alto=720):
    box = f'<iframe src="{url}?embed=1" width="100%" height="{alto}" style="border:0;border-radius:10px;background:#faf7ef"></iframe>'
    box += f'<p><a href="{url}" target="_blank">Abrir tutorial en pantalla completa ↗</a></p>'
    display(HTML(box))

print("Soporte S6 listo.")

---
## 1. Recuperar el proceso que Laura abrió en S5

S5 dejó `s05_ancla_s06.json`. Súbelo al panel **Archivos** de Colab y escribe su ruta. Si lo perdiste, la clase no se bloquea: el dataset trae una **ancla pedagógica real** con historial útil.

**OJO.** El respaldo permite aprender Neo4j, pero el hito declara que no se usó el archivo propio.

In [ ]:
import json
import urllib.request
from pathlib import Path
import pandas as pd

DATA_URL = 'https://raw.githubusercontent.com/jazaineam1/BigData2026/main/Datos/s06_contexto_relacional.csv'
MANIFEST_URL = 'https://raw.githubusercontent.com/jazaineam1/BigData2026/main/Datos/s06_contexto_relacional_manifest.json'

datos = pd.read_csv(DATA_URL, low_memory=False)
with urllib.request.urlopen(MANIFEST_URL) as r:
    manifest = json.loads(r.read().decode("utf-8"))

ruta = input("Ruta de s05_ancla_s06.json (Enter = respaldo): ").strip()
if ruta and Path(ruta).is_file():
    ancla_original = json.loads(Path(ruta).read_text(encoding="utf-8"))
    origen_ancla = "archivo propio S5"
else:
    ancla_original = dict(manifest["ancla_pedagogica"])
    origen_ancla = "ancla pedagógica versionada"

print("Origen:", origen_ancla)
print(json.dumps(ancla_original, ensure_ascii=False, indent=2))
print("Filas disponibles:", len(datos))

### Cómo se lee la entrada

**Cómo se lee.** El ancla identifica un proceso que ya sobrevivió a la regla de S5. El extracto histórico añade hechos adjudicados sin cambiar por qué ese proceso fue priorizado.

**Qué nos dice.** S6 continúa una decisión ya tomada.

**Qué NO permite concluir todavía.** Tener historial contractual no significa que exista una relación problemática.

**Error frecuente.** Volver a construir los 77 candidatos. Eso repetiría S5.

---
## 2. El candidato y el historial cumplen funciones distintas

```text
                    ┌─ Proceso histórico ─→ Proveedor A
                    │
Proceso candidato ← Entidad
                    │
                    └─ Proceso histórico ─→ Proveedor B ─← otra Entidad
```

El candidato puede no estar adjudicado: **no inventamos un proveedor**. El historial adjudicado aporta las relaciones observadas.

In [ ]:
#@title Autoevaluación 1 — Modelo { display-mode: "form" }
pregunta_codificada("eyJudW1lcm8iOiAxLCAidGVtYSI6ICJNb2RlbG8iLCAicHJlZ3VudGEiOiAiwr9Qb3IgcXXDqSBlbCBjYW5kaWRhdG8gZGUgUzUgbm8gbmVjZXNpdGEgdG9kYXbDrWEgdW5hIHJlbGFjacOzbiBoYWNpYSB1biBwcm92ZWVkb3I/IiwgIm9wY2lvbmVzIjogWyJQb3JxdWUgTmVvNGogbm8gc29wb3J0YSBwcm92ZWVkb3JlcyBlbiBwcm9jZXNvcyByZWNpZW50ZXMuIiwgIlBvcnF1ZSBwdWVkZSBubyBlc3RhciBhZGp1ZGljYWRvOyBzaXJ2ZSBjb21vIGFuY2xhIHkgZWwgaGlzdG9yaWFsIGFwb3J0YSBwcm92ZWVkb3JlcyByZWFsZXMuIiwgIlBvcnF1ZSBsb3MgcHJvdmVlZG9yZXMgcGVydGVuZWNlbiBhIEVsYXN0aWNzZWFyY2guIl0sICJjb3JyZWN0YSI6IDEsICJyZXRybyI6IFsiTmVvNGogc8OtIHNvcG9ydGEgZXNhIHJlbGFjacOzbjsgZWwgbMOtbWl0ZSBlc3TDoSBlbiBsYSBldmlkZW5jaWEuIiwgIkV4YWN0by4gTm8gZmFicmljYW1vcyB1bmEgcmVsYWNpw7NuIHF1ZSBlbCBkYXRvIG5vIHNvc3RpZW5lLiIsICJFbGFzdGljc2VhcmNoIHJlc29sdmVyw6Egb3RyYSBwcmVndW50YTogYsO6c3F1ZWRhIHRleHR1YWwgeSByZWxldmFuY2lhLiJdfQ==")

In [ ]:
nit_deseado = str(ancla_original.get("nit_entidad", "")).strip()
hist = datos[datos["tipo_registro"].eq("historico_adjudicado")].copy()
hist_ancla = hist[hist["nit_entidad"].astype(str).str.strip().eq(nit_deseado)]

if hist_ancla.empty:
    print("Tu ancla no tiene historial suficiente en este extracto. Usamos respaldo pedagógico.")
    ancla_trabajo = dict(manifest["ancla_pedagogica"])
    nit_deseado = str(ancla_trabajo["nit_entidad"]).strip()
    hist_ancla = hist[hist["nit_entidad"].astype(str).str.strip().eq(nit_deseado)]
    uso_respaldo_s06 = True
else:
    ancla_trabajo = ancla_original
    uso_respaldo_s06 = origen_ancla != "archivo propio S5"

print("Entidad de trabajo:", ancla_trabajo["entidad"])
print("Procesos históricos:", hist_ancla["id_proceso"].nunique())
print("Proveedores distintos:", hist_ancla["nit_proveedor"].nunique())

### Interpretación del contexto histórico

**Cómo se lee.** Los conteos corresponden al historial adjudicado disponible para la entidad de trabajo, no al proceso candidato aislado.

**Qué nos dice.** Hay material relacional suficiente para preguntar por proveedores y conexiones entre procesos.

**Qué NO permite concluir todavía.** Más procesos o proveedores no equivalen a mayor riesgo. Faltan criterios sobre competencia, temporalidad y comportamiento esperado de la entidad.

**Error frecuente.** Usar el número de contratos como una puntuación de sospecha.

---
## 3. Diseñar el grafo antes de escribir la consulta final

### EJERCICIO S06-PATRON — un solo hueco

Completa **solo** el nombre de la relación entre un proceso histórico y el proveedor al que fue adjudicado.

**Qué debe verse si salió bien:** `(p:Proceso)-[:ADJUDICADO_A]->(v:Proveedor)`.  
**Error probable:** dejar `____` o inventar un verbo que no representa el hecho del dato.  
**Qué significa:** el modelo aún no expresa la semántica contractual que luego recorrerá `MATCH`.

<details><summary><strong>Recuperación si te atascaste</strong></summary>
La relación se llama <code>ADJUDICADO_A</code>. Cámbiala y vuelve a ejecutar.
</details>

In [ ]:
RELACION_PROCESO_PROVEEDOR = "____"  # reemplaza únicamente ____
patron_estudiante = f"(p:Proceso)-[:{RELACION_PROCESO_PROVEEDOR}]->(v:Proveedor)"
print(patron_estudiante)

if RELACION_PROCESO_PROVEEDOR != "ADJUDICADO_A":
    raise ValueError("Revisa el hecho contractual que conecta Proceso con Proveedor.")
print("Patrón correcto: la relación expresa una adjudicación observada.")

### Modelo mínimo que usaremos

```text
(e:Entidad)-[:PUBLICA]->(p:Proceso)-[:ADJUDICADO_A]->(v:Proveedor)
```

| Elemento | Identificador | Decisión |
|---|---|---|
| `Entidad` | NIT | actor que publica |
| `Proceso` | ID SECOP | nodo con texto, valor, modalidad y URL |
| `Proveedor` | NIT | actor adjudicado que puede conectar procesos |
| `PUBLICA` | relación | quién publica el proceso |
| `ADJUDICADO_A` | relación | a quién se adjudicó un proceso histórico |

`Proceso` queda como nodo porque hoy participa en caminos y la siguiente sesión reutilizará su texto.

### Cypher mínimo

| Construcción | Para qué sirve | Qué devuelve/cambia | Error frecuente |
|---|---|---|---|
| `MERGE` | encuentra o crea un patrón | nodos/relaciones persistidos | creer que siempre crea otro nodo |
| `MATCH` | busca patrones | filas con coincidencias | leerlo como un `SELECT *` sin relaciones |
| `WHERE` | filtra | menos coincidencias | filtrar antes de entender el patrón |
| `WITH` | encadena etapas | variables para la etapa siguiente | olvidar qué variables siguen vivas |
| `RETURN` | define la salida | columnas del resultado | confundir salida con persistencia |
| `ORDER BY` / `LIMIT` | ordena y acota | resultado priorizado | asumir orden si no se pidió |

**PARA LLEVAR.** La flecha es parte de la consulta: no es decoración visual.

In [ ]:
#@title Autoevaluación 2 — Cypher { display-mode: "form" }
pregunta_codificada("eyJudW1lcm8iOiAyLCAidGVtYSI6ICJDeXBoZXIiLCAicHJlZ3VudGEiOiAiwr9Qb3IgcXXDqSB1c2FyZW1vcyBNRVJHRSB5IHJlc3RyaWNjaW9uZXMgw7puaWNhcz8iLCAib3BjaW9uZXMiOiBbIlBhcmEgcG9kZXIgcmVwZXRpciBsYSBjYXJnYSBzaW4gZmFicmljYXIgZHVwbGljYWRvcyBkZWwgbWlzbW8gaWRlbnRpZmljYWRvci4iLCAiUG9ycXVlIENSRUFURSBubyBwdWVkZSBjcmVhciByZWxhY2lvbmVzLiIsICJQb3JxdWUgTUVSR0UgZGVjaWRlIGVsIG1vZGVsbyBwb3Igbm9zb3Ryb3MuIl0sICJjb3JyZWN0YSI6IDAsICJyZXRybyI6IFsiQ29ycmVjdG8uIExhIGlkZW50aWRhZCBleHBsw61jaXRhIGhhY2UgbGEgY2FyZ2EgcmVwZXRpYmxlLiIsICJDUkVBVEUgc8OtIHB1ZWRlIGNyZWFyIHJlbGFjaW9uZXMuIiwgIkVsIG1vZGVsbyBzaWd1ZSBzaWVuZG8gdW5hIGRlY2lzacOzbiBodW1hbmEuIl19")

---
## 4. Contrato de resultado: primero pandas

Antes de usar Neo4j calculamos qué proveedores de la entidad ancla también aparecen en otras entidades del extracto. Luego exigiremos a Neo4j la misma respuesta.

In [ ]:
prov_ancla = (
    hist_ancla.groupby(["nit_proveedor", "proveedor"], dropna=False)["id_proceso"]
    .nunique().rename("procesos_con_entidad").reset_index()
)
prov_global = (
    hist.groupby(["nit_proveedor", "proveedor"], dropna=False)["nit_entidad"]
    .nunique().rename("entidades_conectadas").reset_index()
)
esperado_pd = (
    prov_ancla.merge(prov_global, on=["nit_proveedor", "proveedor"], how="left")
    .sort_values(["entidades_conectadas", "procesos_con_entidad", "nit_proveedor"], ascending=[False, False, True])
    .head(10).reset_index(drop=True)
)
esperado_pd

### Interpretación del contrato pandas

**Cómo se lee.** `procesos_con_entidad` cuenta procesos adjudicados de la entidad ancla; `entidades_conectadas` cuenta entidades distintas asociadas al mismo NIT de proveedor.

**Qué nos dice.** Ya sabemos qué salida debería reproducir el grafo.

**Qué NO permite concluir todavía.** Repetición o conectividad no equivale a favorecimiento, colusión ni irregularidad. Faltarían evidencia sobre competencia, temporalidad, propiedad/representación y criterios de adjudicación.

**Error frecuente.** Llamar “sospechoso” al proveedor que queda primero.

### RECUPERACIÓN S06 — si Colab reinició antes de Aura

Ejecuta la siguiente celda siempre que vuelvas del receso. Si el estado sigue vivo, solo lo confirma. Si se perdió, reconstruye datos, ancla de trabajo, historial y contrato pandas.

**OJO.** Después de un reinicio también se recuperan los helpers de las autoevaluaciones y del tutorial. El respaldo pedagógico queda declarado; no se presenta como evidencia propia de S5.

In [ ]:
#@title Recuperar estado S6 { display-mode: "form" }
# RECUPERACIÓN S06
if "pregunta_codificada" not in globals() or "tutorial" not in globals():
    exec('\nimport base64, json, html as html_lib\nfrom IPython.display import display, HTML\n\ndef pregunta_codificada(token):\n    p = json.loads(base64.b64decode(token).decode("utf-8"))\n    uid = f"s06-p{p[\'numero\']}"\n    opts = "".join(\n        f\'<label style="display:block;margin:8px 0"><input type="radio" name="{uid}" value="{i}"> {html_lib.escape(op)}</label>\'\n        for i, op in enumerate(p["opciones"])\n    )\n    retro = json.dumps(p["retro"], ensure_ascii=False)\n    box = (\n        f\'<div style="border:2px solid #175c3c;background:#f4faf6;color:#172019;border-radius:12px;padding:15px;margin:14px 0">\'\n        f\'<strong>Pregunta {p["numero"]} · {html_lib.escape(p["tema"])}</strong>\'\n        f\'<p>{html_lib.escape(p["pregunta"])}</p>{opts}\'\n        f\'<button onclick="(function(){{const e=document.querySelector(\\\'input[name={uid}]:checked\\\');\'\n        f\'const s=document.getElementById(\\\'r-{uid}\\\');if(!e){{s.textContent=\\\'Selecciona una opción.\\\';return;}}\'\n        f\'const i=Number(e.value),r={retro};const ok=i==={p["correcta"]};\'\n        f\'s.innerHTML=\\\'<div style=&quot;margin-top:8px;padding:8px;border-radius:7px;background:#ffffff;color:#172019;border:1px solid #c7d8cd&quot;><strong>\\\'+(ok?\\\'Correcto. \\\':\\\'Revisa. \\\')+\\\'</strong>\\\'+r[i]+\\\'</div>\\\';}})()" \'\n        f\'style="background:#175c3c;color:white;border:0;border-radius:7px;padding:8px 12px">Verificar</button>\'\n        f\'<div id="r-{uid}" aria-live="polite"></div></div>\'\n    )\n    display(HTML(box))\n\ndef tutorial(url, alto=720):\n    box = f\'<iframe src="{url}?embed=1" width="100%" height="{alto}" style="border:0;border-radius:10px;background:#faf7ef"></iframe>\'\n    box += f\'<p><a href="{url}" target="_blank">Abrir tutorial en pantalla completa ↗</a></p>\'\n    display(HTML(box))\n\nprint("Soporte S6 listo.")\n')

estado_necesario = ["datos", "manifest", "ancla_original", "hist", "hist_ancla", "ancla_trabajo", "esperado_pd", "nit_deseado"]
if not all(nombre in globals() for nombre in estado_necesario):
    import json, urllib.request
    from pathlib import Path
    import pandas as pd

    DATA_URL = 'https://raw.githubusercontent.com/jazaineam1/BigData2026/main/Datos/s06_contexto_relacional.csv'
    MANIFEST_URL = 'https://raw.githubusercontent.com/jazaineam1/BigData2026/main/Datos/s06_contexto_relacional_manifest.json'
    datos = pd.read_csv(DATA_URL, low_memory=False)
    with urllib.request.urlopen(MANIFEST_URL) as r:
        manifest = json.loads(r.read().decode("utf-8"))

    ruta_recuperacion = input("Ruta de s05_ancla_s06.json (Enter = respaldo): ").strip()
    if ruta_recuperacion and Path(ruta_recuperacion).is_file():
        ancla_original = json.loads(Path(ruta_recuperacion).read_text(encoding="utf-8"))
        origen_ancla = "archivo propio S5"
    else:
        ancla_original = dict(manifest["ancla_pedagogica"])
        origen_ancla = "ancla pedagógica versionada"

    nit_deseado = str(ancla_original.get("nit_entidad", "")).strip()
    hist = datos[datos["tipo_registro"].eq("historico_adjudicado")].copy()
    hist_ancla = hist[hist["nit_entidad"].astype(str).str.strip().eq(nit_deseado)]
    if hist_ancla.empty:
        ancla_trabajo = dict(manifest["ancla_pedagogica"])
        nit_deseado = str(ancla_trabajo["nit_entidad"]).strip()
        hist_ancla = hist[hist["nit_entidad"].astype(str).str.strip().eq(nit_deseado)]
        uso_respaldo_s06 = True
    else:
        ancla_trabajo = ancla_original
        uso_respaldo_s06 = origen_ancla != "archivo propio S5"

    prov_ancla = (
        hist_ancla.groupby(["nit_proveedor", "proveedor"], dropna=False)["id_proceso"]
        .nunique().rename("procesos_con_entidad").reset_index()
    )
    prov_global = (
        hist.groupby(["nit_proveedor", "proveedor"], dropna=False)["nit_entidad"]
        .nunique().rename("entidades_conectadas").reset_index()
    )
    esperado_pd = (
        prov_ancla.merge(prov_global, on=["nit_proveedor", "proveedor"], how="left")
        .sort_values(["entidades_conectadas", "procesos_con_entidad", "nit_proveedor"], ascending=[False, False, True])
        .head(10).reset_index(drop=True)
    )
    print("Estado S6 reconstruido desde archivos versionados.")
else:
    print("Estado S6 sigue en memoria; no fue necesario reconstruirlo.")

print("Entidad de trabajo:", ancla_trabajo["entidad"])
print("Filas contrato pandas:", len(esperado_pd))

---
## 5. Tutorial visual — AuraDB

**HAZ ESTO AHORA.** Vuelve cuando `RETURN 1 AS conexion` funcione en Query y tengas URI, usuario y contraseña.

El HTML es **instrumental**: muestra el camino de interfaz. Las pantallas dibujadas están rotuladas como representaciones; no se presentan como capturas autenticadas.

In [ ]:
#@title Abrir tutorial Neo4j Aura { display-mode: "form" }
tutorial('https://jazaineam1.github.io/BigData2026/assets/tutoriales/neo4j-aura-s06-paso-a-paso.html')

In [ ]:
!pip install -q "neo4j>=6,<7"
from getpass import getpass
from neo4j import GraphDatabase

URI = input("Connection URI: ").strip()
USER = input("User name: ").strip()
PASSWORD = getpass("Password (no se muestra): ")
if not URI or not USER or not PASSWORD:
    raise ValueError("URI, usuario y contraseña son obligatorios.")

driver = GraphDatabase.driver(URI, auth=(USER, PASSWORD))
driver.verify_connectivity()
print("Conexión Neo4j verificada.")

---
## 6. Identidad y carga idempotente

Primero creamos restricciones. Después `UNWIND` recibe una lista de filas desde Python y `MERGE` reutiliza nodos ya existentes.

**Qué debe verse:** tres restricciones válidas y una carga que puede repetirse sin multiplicar el mismo NIT/ID.  
**Error probable:** autenticación o conectividad antes de ejecutar Cypher. Eso es un problema instrumental, no un problema del modelo; usa el diagnóstico del tutorial.

In [ ]:
constraints = [
    "CREATE CONSTRAINT entidad_nit IF NOT EXISTS FOR (e:Entidad) REQUIRE e.nit IS UNIQUE",
    "CREATE CONSTRAINT proceso_id IF NOT EXISTS FOR (p:Proceso) REQUIRE p.id IS UNIQUE",
    "CREATE CONSTRAINT proveedor_nit IF NOT EXISTS FOR (v:Proveedor) REQUIRE v.nit IS UNIQUE",
]
for q in constraints:
    driver.execute_query(q)
print("Restricciones listas.")

In [ ]:
cols = [
    "entidad", "nit_entidad", "departamento_entidad", "id_proceso", "referencia",
    "nombre_proceso", "descripcion", "precio_base", "modalidad", "proveedor",
    "nit_proveedor", "departamento_proveedor", "noticias_entidad", "nivel_menciones",
    "url_secop", "es_proceso_candidato_s05", "es_entidad_candidata_s05",
]
rows = datos[cols].where(pd.notna(datos[cols]), None).to_dict("records")

query_base = '''
UNWIND $filas AS fila
MERGE (e:Entidad {nit: toString(fila.nit_entidad)})
SET e.nombre = fila.entidad,
    e.departamento = fila.departamento_entidad,
    e.es_candidata_s05 = fila.es_entidad_candidata_s05,
    e.noticias_entidad = fila.noticias_entidad,
    e.nivel_menciones = fila.nivel_menciones
MERGE (p:Proceso {id: fila.id_proceso})
SET p.referencia = fila.referencia,
    p.nombre = fila.nombre_proceso,
    p.descripcion = fila.descripcion,
    p.valor = fila.precio_base,
    p.modalidad = fila.modalidad,
    p.url = fila.url_secop,
    p.es_candidato_s05 = fila.es_proceso_candidato_s05
MERGE (e)-[:PUBLICA]->(p)
'''
driver.execute_query(query_base, filas=rows)

rows_proveedor = [r for r in rows if r.get("nit_proveedor")]
query_proveedor = '''
UNWIND $filas AS fila
MATCH (p:Proceso {id: fila.id_proceso})
MERGE (v:Proveedor {nit: toString(fila.nit_proveedor)})
SET v.nombre = fila.proveedor, v.departamento = fila.departamento_proveedor
MERGE (p)-[:ADJUDICADO_A]->(v)
'''
driver.execute_query(query_proveedor, filas=rows_proveedor)
print("Carga lista:", len(rows), "filas;", len(rows_proveedor), "adjudicaciones.")

---
## 7. La consulta que justifica Neo4j

Ahora recorremos el patrón Entidad → Proceso → Proveedor y, desde ese proveedor, contamos otras entidades conectadas.

In [ ]:
query_contexto = '''
MATCH (e:Entidad {nit:$nit})-[:PUBLICA]->(p:Proceso)-[:ADJUDICADO_A]->(v:Proveedor)
WITH v, count(DISTINCT p) AS procesos_con_entidad
MATCH (otra:Entidad)-[:PUBLICA]->(:Proceso)-[:ADJUDICADO_A]->(v)
RETURN v.nit AS nit_proveedor,
       v.nombre AS proveedor,
       procesos_con_entidad,
       count(DISTINCT otra) AS entidades_conectadas
ORDER BY entidades_conectadas DESC, procesos_con_entidad DESC, nit_proveedor ASC
LIMIT 10
'''
neo = driver.execute_query(query_contexto, nit=nit_deseado)
neo_df = pd.DataFrame([r.data() for r in neo.records])
neo_df

In [ ]:
cols_cmp = ["nit_proveedor", "procesos_con_entidad", "entidades_conectadas"]
pd_cmp = esperado_pd[cols_cmp].copy()
neo_cmp = neo_df[cols_cmp].copy()
pd_cmp["nit_proveedor"] = pd_cmp["nit_proveedor"].astype(str)
neo_cmp["nit_proveedor"] = neo_cmp["nit_proveedor"].astype(str)
coinciden = pd_cmp.reset_index(drop=True).equals(neo_cmp.reset_index(drop=True))
print("pandas == Neo4j:", coinciden)
assert coinciden, "La respuesta Neo4j no coincide con el contrato pandas."

### Interpretación pandas ↔ Neo4j

**Cómo se lee.** Comparamos NIT y las dos métricas en el mismo orden.

**Qué nos dice.** El grafo reproduce el patrón calculado previamente.

**Qué NO permite concluir todavía.** Es una prueba de corrección, no un benchmark de velocidad ni evidencia de irregularidad.

**Error frecuente.** Confundir “la consulta coincide” con “Neo4j es más rápido”.

In [ ]:
#@title Autoevaluación 3 — Interpretación { display-mode: "form" }
pregunta_codificada("eyJudW1lcm8iOiAzLCAidGVtYSI6ICJJbnRlcnByZXRhY2nDs24iLCAicHJlZ3VudGEiOiAiVW4gcHJvdmVlZG9yIGFwYXJlY2UgY29uZWN0YWRvIGNvbiBjdWF0cm8gZW50aWRhZGVzLiDCv1F1w6kgcHVlZGUgYWZpcm1hciBMYXVyYT8iLCAib3BjaW9uZXMiOiBbIlF1ZSBleGlzdGUgdW5hIHJlbGFjacOzbiBjb250cmFjdHVhbCBvYnNlcnZhZGEgY29uIHByb2Nlc29zIGRlIGN1YXRybyBlbnRpZGFkZXMgZGVudHJvIGRlbCBleHRyYWN0by4iLCAiUXVlIGxhcyBjdWF0cm8gZW50aWRhZGVzIGNvb3JkaW5hcm9uIHN1cyBhZGp1ZGljYWNpb25lcy4iLCAiUXVlIGVsIHByb3ZlZWRvciBpbmN1cnJpw7MgZW4gdW5hIGlycmVndWxhcmlkYWQuIl0sICJjb3JyZWN0YSI6IDAsICJyZXRybyI6IFsiQ29ycmVjdG8uIEVsIGdyYWZvIGRlc2NyaWJlIGVzdHJ1Y3R1cmEgcmVnaXN0cmFkYS4iLCAiTGEgY29uZWN0aXZpZGFkIHBvciBzw60gc29sYSBubyBwcnVlYmEgY29vcmRpbmFjacOzbi4iLCAiTGEgY29uZWN0aXZpZGFkIHBvciBzw60gc29sYSBubyBwcnVlYmEgaXJyZWd1bGFyaWRhZC4iXX0=")

---
## 8. CRUD seguro y evidencia individual

El CRUD usa `S06-DEMO`; no modificamos un proceso real. Después eliges un proveedor de **tu resultado** y abres su vecindario.

**Qué debe verse:** una tabla con entidades y procesos relacionados con el proveedor elegido.  
**Error probable:** escoger un número fuera del top mostrado. Significa que tu decisión no corresponde al resultado ejecutado.  
**Recuperación:** vuelve a ejecutar y elige un número de la lista; no inventes un NIT.

In [ ]:
driver.execute_query('''
MERGE (e:Entidad {nit:'S06-E'}) SET e.nombre='Entidad demo'
MERGE (p:Proceso {id:'S06-DEMO'}) SET p.nombre='Proceso demo'
MERGE (v:Proveedor {nit:'S06-V'}) SET v.nombre='Proveedor demo'
MERGE (e)-[:PUBLICA]->(p)
MERGE (p)-[:ADJUDICADO_A]->(v)
''')
r = driver.execute_query("MATCH (p:Proceso {id:'S06-DEMO'}) SET p.estado_revision='revisado' RETURN p.estado_revision AS estado")
assert r.records[0]["estado"] == "revisado"
driver.execute_query("MATCH (n) WHERE n.nit IN ['S06-E','S06-V'] OR n.id='S06-DEMO' DETACH DELETE n")
print("CRUD demo completado y limpiado.")

In [ ]:
if neo_df.empty:
    raise ValueError("No hay proveedores para elegir.")
for i, row in neo_df.iterrows():
    print(f"{i+1:>2}. {row['proveedor']} | entidades={row['entidades_conectadas']}")
sel = int(input("Número de proveedor: ").strip())
if not 1 <= sel <= len(neo_df):
    raise ValueError("Número fuera de rango")
proveedor_elegido = neo_df.iloc[sel-1]

vec = driver.execute_query('''
MATCH (e:Entidad)-[:PUBLICA]->(p:Proceso)-[:ADJUDICADO_A]->(v:Proveedor {nit:$nit})
RETURN e.nombre AS entidad, p.id AS proceso, p.nombre AS nombre_proceso, p.valor AS valor
ORDER BY entidad, valor DESC
''', nit=str(proveedor_elegido["nit_proveedor"]))
vecindario_df = pd.DataFrame([r.data() for r in vec.records])
vecindario_df

### Interpretación de tu vecindario

**Cómo se lee.** Cada fila es un proceso conectado al proveedor que elegiste; una misma entidad puede aportar varios procesos.

**Qué nos dice.** Puedes observar qué entidades y procesos del extracto comparten ese actor contractual y abrir casos concretos para revisión.

**Qué NO permite concluir todavía.** Compartir proveedor no demuestra coordinación, favorecimiento ni irregularidad. Faltan, como mínimo, cronología comparable, condiciones de competencia y vínculos de propiedad/representación cuando la hipótesis los requiera.

**Error frecuente.** Convertir el número de conexiones en un “score de riesgo” sin modelo ni denominador.

### La evidencia no termina en el grafo

Ahora registra dos decisiones que una respuesta genérica no puede inventar por ti:

1. un límite que nombre **qué dato faltaría** antes de una afirmación de riesgo/irregularidad;
2. una alternativa de modelado que descartaste y por qué.

In [ ]:
from pathlib import Path
limite_estudiante = input("Límite concreto y dato faltante: ").strip()
alternativa_modelo = input("Alternativa de modelado descartada: ").strip()
razon_alternativa = input("¿Por qué la descartaste para esta pregunta?: ").strip()

if len(limite_estudiante) < 25:
    raise ValueError("Nombra la conclusión que no puedes sostener y el dato que falta.")
if len(alternativa_modelo) < 5 or len(razon_alternativa) < 15:
    raise ValueError("Nombra una alternativa real y explica por qué no sirve igual de bien para esta pregunta.")

export = vecindario_df.merge(
    datos[["id_proceso", "descripcion", "modalidad", "url_secop"]].drop_duplicates("id_proceso"),
    left_on="proceso", right_on="id_proceso", how="left"
)
export.to_json("s06_contexto_procesos.jsonl", orient="records", lines=True, force_ascii=False)

hito = f'''# Hito S06 — Ficha relacional de revisión

- Origen del ancla: {origen_ancla}
- Proceso elegido en S5: {ancla_original.get("id_proceso", "")}
- Proceso/entidad usados para el grafo: {ancla_trabajo.get("id_proceso", "")} — {ancla_trabajo.get("entidad", "")}
- Noticias / nivel: {ancla_trabajo.get("noticias_entidad", "")} / {ancla_trabajo.get("nivel_menciones", "")}
- Respaldo pedagógico: {uso_respaldo_s06}
- pandas == Neo4j: {coinciden}
- Proveedor elegido: {proveedor_elegido["proveedor"]}
- Entidades conectadas: {int(proveedor_elegido["entidades_conectadas"])}
- Procesos en el vecindario: {len(vecindario_df)}

## Límite
{limite_estudiante}

## Decisión de modelado
Proceso se modeló como nodo porque participa en caminos y su texto será reutilizado en la siguiente sesión.

### Alternativa descartada
{alternativa_modelo}

Razón: {razon_alternativa}
'''
Path("hito_s06_ficha_relacional.md").write_text(hito, encoding="utf-8")
print(hito)

try:
    from google.colab import files
    files.download("hito_s06_ficha_relacional.md")
    files.download("s06_contexto_procesos.jsonl")
except Exception:
    print("Archivos generados en el runtime.")

## Rúbrica S06

| Criterio | Completo | Parcial | Sin evidencia | Peso |
|---|---|---|---|---:|
| Continuidad | identifica proceso S5 y declara respaldo | solo entidad | no conecta con S5 | 15 |
| Modelo | justifica nodos/relaciones + alternativa descartada | describe sin alternativa | copia el patrón | 20 |
| Ejecución | vecindario propio ejecutado | solo consulta común | no hay salida | 20 |
| Verificación | `pandas == Neo4j` comprobado | muestra ambos | solo uno | 15 |
| Evidencia propia | proveedor + entidades + procesos | incompleta | genérica | 15 |
| Límite | conclusión inválida + dato específico faltante | genérico | afirma irregularidad | 15 |

Las autoevaluaciones son formativas. El hito es la evidencia revisable de la sesión.

---
## Hoja de trucos y puente

```text
MERGE  → encuentra o crea
MATCH  → busca patrón
WHERE  → filtra
WITH   → encadena
RETURN → salida

Entidad -PUBLICA-> Proceso -ADJUDICADO_A-> Proveedor
```

**Idea central.** Cassandra organizó datos para una pregunta repetitiva conocida. Neo4j hace de las relaciones una parte explícita de la pregunta.

### Lo que sigue

Laura ya puede ver el vecindario, pero ahora tiene muchos nombres y descripciones de procesos. La nueva pregunta será:

> **¿Cuáles de esos procesos son más relevantes para una búsqueda textual concreta?**

`s06_contexto_procesos.jsonl` será la entrada de Elasticsearch/BM25.

In [ ]:
try:
    driver.close()
    print("Conexión Neo4j cerrada.")
except Exception:
    pass